# Lyrics Lens 主分析流程
從原始歌詞到情緒模型比較。沿用原專案 pandas、TF-IDF、K-means、VADER 與 DistilBERT 方法；先修正資料與比較條件。

**只需要這份 notebook 和 songs_with_raw_lyrics.csv 放在同一資料夾。** 輸出統一寫入 results。先執行第 0–4 節，檢查結果後才跑模型。
此 notebook 尚未在你的電腦執行。提供者已驗證清理步驟及程式語法，沒有宣稱已跑完語言/情緒模型。
原始歌詞不覆寫；年度是專輯版本年，尚非首次發行年。網站不應直接公開完整歌詞。


## 0 安裝套件
在目前 notebook 的 Python 環境安裝。首次需要網路；若已有套件可略過。模型稍後仍需下載。

In [ ]:
%pip install pandas numpy scikit-learn nltk transformers torch langdetect

## 1 讀取原始 CSV
不再呼叫歌詞 API。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, json, unicodedata, hashlib

INPUT = Path("songs_with_raw_lyrics.csv")
OUT = Path("results")
OUT.mkdir(exist_ok=True)
assert INPUT.exists(), "請把 songs_with_raw_lyrics.csv 放在 notebook 旁"
raw = pd.read_csv(INPUT, encoding="utf-8-sig")
assert {"lyrics_raw", "Track URI", "ISRC", "year"}.issubset(raw.columns)
print("原始紀錄：", len(raw))


## 2 沿用並整合先前清理
以下兩個 cell 是 Step1、Step2 的函數，現在直接包含於本 notebook，不需要另外的 .py 檔。保留中間表供追查；run 名稱重新定義後立即使用，不會要求從其他資料夾 import。

In [ ]:
"""Run: python clean_lyrics.py --input songs_with_raw_lyrics.csv --output results
Only dependency: pandas. No API calls or model inference.
"""
from pathlib import Path
import argparse, re, unicodedata, hashlib, json
import pandas as pd

def clean_sentiment(text):
    text = unicodedata.normalize('NFC', text).replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'\[[^\]\n]*\]', '', text)  # bracketed section/singer labels
    lines = [re.sub(r'[^\S\n]+', ' ', x).strip() for x in text.split('\n')]
    return '\n'.join(x for x in lines if x)

def run(input_path, output_path):
    out = Path(output_path); out.mkdir(parents=True, exist_ok=True)
    f = pd.read_csv(input_path, encoding='utf-8-sig')
    assert {'lyrics_raw', 'Track URI', 'ISRC', 'year'}.issubset(f.columns)
    f.insert(0, 'source_row_id', range(1, len(f)+1))
    raw = f.lyrics_raw.fillna('')
    f['lyrics_sentiment'] = raw.map(clean_sentiment)
    # Preserve negation/contractions; stopwords are chosen later for vectorization only.
    f['lyrics_tfidf'] = f.lyrics_sentiment.str.lower().str.replace(r'[^a-z\s\u0027\u2019]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
    f['lyrics_unique_lines'] = f.lyrics_sentiment.map(lambda t: '\n'.join(dict.fromkeys(t.splitlines())))
    f['has_text'] = f.lyrics_sentiment.str.strip().ne('')
    f['raw_char_count'] = raw.str.len()
    f['word_count'] = f.lyrics_sentiment.str.split().str.len()
    f['line_count'] = f.lyrics_sentiment.map(lambda t: len(t.splitlines()))
    f['unique_line_count'] = f.lyrics_sentiment.map(lambda t: len(set(x.casefold() for x in t.splitlines())))
    f['repeat_line_ratio'] = 1 - f.unique_line_count / f.line_count.replace(0, float('nan'))
    f['lyric_hash'] = f.lyrics_sentiment.map(lambda t: hashlib.sha256(t.encode()).hexdigest() if t else '')
    valid_uri = f['Track URI'].notna() & f['Track URI'].astype(str).str.strip().ne('')
    f['duplicate_uri_extra'] = valid_uri & f['Track URI'].duplicated()
    f['duplicate_uri_group'] = valid_uri & f['Track URI'].duplicated(False)
    f['duplicate_isrc_group'] = f.ISRC.notna() & f.ISRC.duplicated(False)
    f['duplicate_lyrics_group'] = f.has_text & f.lyric_hash.duplicated(False)
    # Screening only: script is not language. French/Spanish etc. can use Latin letters too.
    def nonlatin(t):
        letters = [c for c in t if c.isalpha()]
        return sum('LATIN' not in unicodedata.name(c, '') for c in letters)/len(letters) if letters else float('nan')
    f['non_latin_letter_ratio'] = f.lyrics_sentiment.map(nonlatin)
    f['language_status'] = 'not_verified'
    f.loc[~f.has_text, 'language_status'] = 'missing_text'
    f['review_reasons'] = f.apply(lambda r: '; '.join(k for k,v in {
        'missing_text':not r.has_text,
        'duplicate_uri':r.duplicate_uri_group,
        'duplicate_isrc':r.duplicate_isrc_group,
        'duplicate_lyrics':r.duplicate_lyrics_group,
        'short_text_under_100_words':r.has_text and r.word_count<100,
        'high_repetition_over_50pct':r.repeat_line_ratio>0.5,
        'non_latin_script_over_5pct':r.non_latin_letter_ratio>0.05,
        'possible_web_metadata':bool(re.search(r'https?://|\bEmbed\b|\b\d+ Contributors\b',r.lyrics_sentiment))
    }.items() if v), axis=1)
    # Candidate != validated English corpus. Do not use directly for final claims.
    candidate = f[f.has_text & ~f.duplicate_uri_extra].copy()
    review = f[f.review_reasons.ne('')].copy()
    review['manual_language'] = ''
    review['correct_song_match'] = ''
    review['review_decision'] = ''
    review['review_notes'] = ''
    for name, frame in [('songs_audited.csv',f),('songs_candidate.csv',candidate),('manual_review.csv',review)]:
        frame.to_csv(out/name,index=False,encoding='utf-8-sig')
    annual=f.groupby('year').agg(selected=('source_row_id','size'),with_text=('has_text','sum'),duplicate_uri_extra=('duplicate_uri_extra','sum'))
    annual.to_csv(out/'annual_coverage.csv',encoding='utf-8-sig')
    stats={'input_rows':len(f),'nonempty_cleaned_text':int(f.has_text.sum()),'missing_text':int((~f.has_text).sum()),'duplicate_uri_extra':int(f.duplicate_uri_extra.sum()),'candidate_rows':len(candidate),'review_rows':len(review),'raw_over_512_characters':int((f.raw_char_count>512).sum()),'all_languages_verified':False,'models_rerun':False}
    (out/'audit_summary.json').write_text(json.dumps(stats,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(stats,ensure_ascii=False,indent=2))
    return f,candidate,review


audited, _, _ = run(INPUT, OUT / "audit")


In [ ]:
from pathlib import Path
import pandas as pd
import re, json

def run(source='results/songs_audited.csv', output='results_step2'):
    f=pd.read_csv(source,encoding='utf-8-sig');out=Path(output);out.mkdir(parents=True,exist_ok=True)
    # Preserve original and step 1 fields. Only edit a new sentiment column.
    f['lyrics_sentiment_v2']=f.lyrics_sentiment.fillna('')
    f['content_edit']=''
    for i,r in f.iterrows():
        t=r.lyrics_sentiment_v2
        if r['Track Name']=='Uprising' and r.artist_clean=='Muse':
            t=re.sub(r'(?m)^\{ From: https?://[^\n]+\}\s*\n?', '', t)
        if r['Track Name']=='Blue Monday - 2016 Remaster' and r.artist_clean=='New Order':
            t='\n'.join(x for x in t.splitlines() if not (x.startswith('Blue Monday*') or x=='FIFA2005原声'))
        if t!=r.lyrics_sentiment_v2:
            f.at[i,'lyrics_sentiment_v2']=t;f.at[i,'content_edit']='removed_observed_non_lyric_metadata'
    f['lyrics_tfidf_v2']=f.lyrics_sentiment_v2.str.lower().str.replace(r'[^a-z\s\u0027\u2019]',' ',regex=True).str.replace(r'\s+',' ',regex=True).str.strip()
    def unique(t):
        seen=set();result=[]
        for line in t.splitlines():
            k=line.casefold()
            if k not in seen:seen.add(k);result.append(line)
        return '\n'.join(result)
    f['lyrics_unique_lines_v2']=f.lyrics_sentiment_v2.map(unique)
    f['word_count_v2']=f.lyrics_sentiment_v2.str.split().str.len()
    def ratio(t):
        lines=t.splitlines()
        return 1-len(set(x.casefold() for x in lines))/len(lines) if lines else float('nan')
    f['repeat_line_ratio_v2']=f.lyrics_sentiment_v2.map(ratio)
    f['record_decision']='keep_candidate'
    f.loc[~f.has_text,'record_decision']='exclude_missing'
    f.loc[f.duplicate_uri_extra,'record_decision']='exclude_duplicate_uri'
    f['duplicate_of_source_row_id']=pd.Series(pd.NA,index=f.index,dtype='Int64')
    eligible=f[f.record_decision.eq('keep_candidate')&f.ISRC.notna()]
    for _,g in eligible.groupby(['year','ISRC','artist_clean','lyric_hash'],sort=False):
        if len(g)>1:
            # Highest supplied Popularity, then stable original row ID. Not a quality judgement.
            g=g.sort_values(['Popularity','source_row_id'],ascending=[False,True],kind='stable')
            keeper=int(g.iloc[0].source_row_id)
            for i in g.index[1:]:
                f.at[i,'record_decision']='merge_duplicate_same_recording_same_text'
                f.at[i,'duplicate_of_source_row_id']=keeper
    f['assessment_note']='No external lyric-reference verification; language not verified.'
    f.loc[f.review_reasons.fillna('').str.contains('high_repetition'),'assessment_note']='Keep repetition; compare full vs unique-line text in later sensitivity analysis.'
    f.loc[f.review_reasons.fillna('').str.contains('short_text'),'assessment_note']='Short text is not proof of truncation; verify completeness against a permitted reference.'
    f.loc[f.source_row_id.eq(138),'assessment_note']='Removed two observed metadata lines; remaining lyrics may be incomplete: verify before modelling.'
    f.loc[f.source_row_id.eq(525),'assessment_note']='Some lines appear questionable; external lyric match/completeness check required. No correction guessed.'
    f['requires_text_verification']=f.source_row_id.isin([29,47,138,405,422,525,686])
    conflicts=[]
    for isrc,g in f[f.has_text&f.ISRC.notna()].groupby('ISRC'):
        if len(g)>1 and g.lyric_hash.nunique()>1:conflicts.extend(g.source_row_id.tolist())
    f['same_isrc_text_conflict']=f.source_row_id.isin(conflicts)
    candidate=f[f.record_decision.eq('keep_candidate')].copy()
    pending=f[f.same_isrc_text_conflict|f.requires_text_verification].copy()
    for c in ['verified_language','verified_song_match','verified_completeness','reference_url','human_notes']:pending[c]=''
    for name,frame in [('songs_reviewed.csv',f),('songs_candidate_step2.csv',candidate),('priority_review.csv',pending),('record_decisions.csv',f[['source_row_id','Track Name','artist_clean','year','record_decision','duplicate_of_source_row_id','content_edit','assessment_note']])]:frame.to_csv(out/name,index=False,encoding='utf-8-sig')
    stats={'original_rows':len(f),'candidate_rows':len(candidate),'record_decisions':f.record_decision.value_counts().to_dict(),'metadata_rows_edited':int(f.content_edit.ne('').sum()),'priority_review_rows':len(pending),'language_verified':False,'external_song_match_verified':False,'model_inference_performed':False}
    (out/'summary.json').write_text(json.dumps(stats,ensure_ascii=False,indent=2),encoding='utf-8');print(json.dumps(stats,indent=2));return f,candidate,pending


reviewed, candidate, priority = run(OUT / "audit/songs_audited.csv", OUT / "record_review")


## 3 分析用去重與品質排除
探索單位設定為錄音：先按 ISRC 去重，再把相同歌手且完全相同的歌詞合併。相同 ISRC 有不同文本時，不宣稱哪份逐字正確：一般依現有 Popularity、source_row_id 決定代表；Grenade 指定保留 source_row_id=360（主要段落核對，仍未逐字驗證）。跨年錄音不重複計權；保留的 year 仍只代表所選版本，不能用作歷史趨勢結論。
暫停使用 source_row_id=138（Blue Monday，缺少參考歌詞後段）、525（Turn Down for What，文字錯誤）。不補寫版權歌詞；其餘短歌詞不因短就自動刪除。這不是完整外部歌詞校驗。
來源：Shazam https://www.shazam.com/song/439807777/blue-monday 、Amazon Music https://music.amazon.com/tracks/B00HFEC192 、LINE MUSIC https://music-tw.line.me/track/1056159001 。查核日期 2026-10-27。


In [ ]:
work = candidate.copy()
work["analysis_decision"] = "keep"
work.loc[work.source_row_id.isin([138, 525]), "analysis_decision"] = "exclude_known_text_quality"
work["representative_preference"] = work.source_row_id.eq(360).astype(int)
work = work.sort_values(["representative_preference", "Popularity", "source_row_id"], ascending=[False, False, True], kind="stable")
eligible = work.analysis_decision.eq("keep")
nonempty_isrc = work.ISRC.notna() & work.ISRC.astype(str).str.strip().ne("")
dup = work.loc[eligible & nonempty_isrc, "ISRC"].duplicated()
work.loc[dup[dup].index, "analysis_decision"] = "exclude_same_isrc"
# Do not merge covers by different singers merely because lyrics match.
eligible = work.analysis_decision.eq("keep")
dup = work.loc[eligible].duplicated(["artist_clean", "lyrics_sentiment_v2"])
work.loc[dup[dup].index, "analysis_decision"] = "exclude_same_artist_same_text"
work = work.sort_values("source_row_id")
work.to_csv(OUT / "analysis_decisions.csv", index=False, encoding="utf-8-sig")
print(work.analysis_decision.value_counts())


## 4 語言初篩（先執行到這裡）
langdetect 是統計模型：結果是初篩，不是人工驗證。對整首與前/後半部辨識，三者均為英文、整首估計值至少 0.90 才標為 likely_english；其餘留待核對。0.90 是工作用門檻，不是已校準準確率；混合語言、短歌詞及重複句可能誤判。
不要用 ASCII 字母比例取代語言識別。保留判別失敗狀態，不默認英文。


In [ ]:
from langdetect import detect_langs, DetectorFactory
DetectorFactory.seed = 1012

def language_screen(text):
    words = text.split()
    try:
        full = detect_langs(text)[0]
        midpoint = len(words) // 2
        first = detect_langs(" ".join(words[:midpoint]))[0].lang
        second = detect_langs(" ".join(words[midpoint:]))[0].lang
        likely = full.lang == first == second == "en" and full.prob >= 0.90
        return pd.Series([full.lang, full.prob, first, second, "likely_english" if likely else "needs_language_review"])
    except Exception as exc:
        return pd.Series([None, None, None, None, "detection_failed"])

screened = work[work.analysis_decision.eq("keep")].copy()
screened[["detected_language", "language_estimate", "first_half_language", "second_half_language", "language_screen"]] = screened.lyrics_sentiment_v2.apply(language_screen)
screened["human_language_verified"] = False
screened.to_csv(OUT / "language_screening.csv", index=False, encoding="utf-8-sig")
print(screened.language_screen.value_counts())
display(screened.loc[screened.language_screen.ne("likely_english"), ["source_row_id", "Track Name", "artist_clean", "detected_language", "language_estimate", "language_screen"]])


In [ ]:
# 使用者已完成的語言初篩與人工閱讀核對（AI 協助）
english_ids = [73,140,199,269,310,379,444,476,538,621,649,690,720,755,833,879,881]
screened["language_review"] = "not_manually_reviewed"
screened.loc[screened.source_row_id.isin(english_ids), "language_review"] = "english_accepted"
screened["include_english"] = (screened.language_screen.eq("likely_english") | screened.language_review.eq("english_accepted")) & screened.source_row_id.ne(533)
# 533 Trumpets 已確認錯配；不重跑其他歌曲的情緒模型。
screened.to_csv(OUT / "language_screening_reviewed.csv", index=False, encoding="utf-8-sig")
print("目前納入：", screened.include_english.sum())


**先停在此處，把語言初篩的數量與待核對表提供給我。** 不需再上傳完整歌詞。後面是準備好的模型流程，等確認初篩後執行。若結果全是 detection_failed，先檢查環境，不要繼續。
## 5 建立同範圍的模型輸入
VADER 與 Transformer 都分析相同的 token 分塊；每首歌的全部 token 都被涵蓋，不只前 512 字元。384 tokens 是保守分塊大小，非重疊分塊避免重複加權；可能切開句子，仍需列為限制。
每首歌用 token 數加權平均分塊分數。兩種分數定義不同，不能把差值當成準確度。另比較原歌詞與忽略大小寫去重行版本。這是轉錄文字分析，無法評估音樂聽感。


In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import pipeline

nltk.download("vader_lexicon")
vader = SentimentIntensityAnalyzer()
MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
REVISION = "714eb0f"  # original notebook's recorded revision; pin for reproducibility
classifier = pipeline("sentiment-analysis", model=MODEL, revision=REVISION, device=-1)
tokenizer = classifier.tokenizer
analysis = screened[screened["include_english"]].copy()
assert len(analysis), "沒有通過初篩的資料；先檢查語言結果"

def score_song(text):
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    assert ids, "空白文本不可分析"
    parts = [ids[i:i+384] for i in range(0, len(ids), 384)]
    texts = [tokenizer.decode(p, skip_special_tokens=True, clean_up_tokenization_spaces=False) for p in parts]
    # DistilBERT tokenizer is uncased; decoding loses case, so BOTH models receive
    # the same decoded lowercase chunks. VADER's capitalization feature is unavailable.
    predictions = classifier(texts, batch_size=8, truncation=False)
    rows = []
    for part, chunk, prediction in zip(parts, texts, predictions):
        label = prediction["label"].upper()
        assert label in {"POSITIVE", "NEGATIVE"}
        p_pos = prediction["score"] if label == "POSITIVE" else 1-prediction["score"]
        rows.append({"tokens":len(part), "vader_compound":vader.polarity_scores(chunk)["compound"], "transformer_p_positive":p_pos})
    detail = pd.DataFrame(rows)
    return {"token_count":len(ids), "chunk_count":len(parts), "vader_score":float(np.average(detail.vader_compound, weights=detail.tokens)), "transformer_score":float(np.average(2*detail.transformer_p_positive-1, weights=detail.tokens))}, rows


## 6 模型執行與續跑
首次需下載模型且會較慢。每首歌每個版本算完就寫入 checkpoint，重跑會跳過同一設定與文本的既有結果。變更模型或文本會產生不同 cache key。不要手動刪 checkpoint，除非要重新計算。

In [ ]:
CACHE = OUT / "sentiment_checkpoint.jsonl"
config = {"model":MODEL, "revision":REVISION, "chunk_size":384, "aggregation":"token_weighted", "input":"uncased_tokenizer_decoded_v1"}
config_text = json.dumps(config, sort_keys=True)
existing = {}
if CACHE.exists():
    for line in CACHE.read_text(encoding="utf-8").splitlines():
        try:
            record = json.loads(line)
            existing[record["cache_key"]] = record
        except json.JSONDecodeError:
            pass  # interrupted trailing record; next successful append starts on a new line
results = []
for n, (_, row) in enumerate(analysis.iterrows(), 1):
    for version, column in [("full", "lyrics_sentiment_v2"), ("unique_lines", "lyrics_unique_lines_v2")]:
        text = row[column]
        key = hashlib.sha256((config_text + text).encode()).hexdigest()
        if key not in existing:
            scores, chunks = score_song(text)
            record = {"cache_key":key, "config":config, "scores":scores, "chunks":chunks}
            with CACHE.open("a", encoding="utf-8") as handle:
                handle.write("\n" + json.dumps(record, ensure_ascii=False) + "\n")
            existing[key] = record
        results.append({"source_row_id":int(row.source_row_id), "text_version":version, **existing[key]["scores"]})
    if n % 25 == 0:
        print(f"已完成 {n}/{len(analysis)} 首")
scores = pd.DataFrame(results)
scores.to_csv(OUT / "sentiment_scores.csv", index=False, encoding="utf-8-sig")
print("已保存情緒結果：", len(scores), "筆（每首兩個文本版本）")


## 7 比較結果與限制
Transformer score 是分塊 (2 × P(positive) − 1) 的加權平均，不是已校準的情緒強度。VADER compound 是另一種規則分數，兩者不可解讀為同一量尺。顯示相關係數及分類一致率只描述一致程度，不代表準確度。VADER 正負門檻採 ±0.05；Transformer 是二元模型，沒有中性類，VADER 中性率須另報。
未有人工作為參考標籤，因此不報 Accuracy 或 F1，也不宣稱 Transformer 比較準。人工評估需要後續另行建立。


In [ ]:
full = scores[scores.text_version.eq("full")].copy()
full["vader_label"] = np.select([full.vader_score >= 0.05, full.vader_score <= -0.05], ["positive", "negative"], default="neutral")
full["transformer_label"] = np.where(full.transformer_score >= 0, "positive", "negative")
print("分數相關（不是準確度）：")
print(full[["vader_score", "transformer_score"]].corr())
print("VADER 中性比例：", full.vader_label.eq("neutral").mean())
nonneutral = full[full.vader_label.ne("neutral")]
print("VADER 非中性子集的兩模型一致率：", (nonneutral.vader_label == nonneutral.transformer_label).mean())
metadata = analysis[["source_row_id", "Track Name", "artist_clean", "year"]]
comparison = full.merge(metadata, on="source_row_id", validate="one_to_one")
comparison.to_csv(OUT / "model_comparison.csv", index=False, encoding="utf-8-sig")
sensitivity = scores.pivot(index="source_row_id", columns="text_version", values=["vader_score", "transformer_score"])
sensitivity.to_csv(OUT / "repetition_sensitivity.csv", encoding="utf-8-sig")
display(comparison[comparison.vader_label.ne(comparison.transformer_label)].head(10))


## 8 TF-IDF 與 K-means 分群（新增，可直接執行本節）
已保存第 7 節結果時，不必重跑模型。以下從 results/model_comparison.csv 及 results/record_review/songs_reviewed.csv 讀取，並再次排除 533。

以去重複行歌詞分析詞彙主題；不同於情緒分析用途，移除英文停用詞與擬聲詞。TF-IDF 採單詞及雙詞、min_df=5、max_df=0.8、最多 3000 特徵、sublinear_tf=True。實際留下 2570 特徵。
這是 K-means 的探索基線，不是主題真值或經人工驗證的歌曲分類。它使用歐氏距離擬合，另用 cosine silhouette 評估，兩者需明確區分。
比較 K=2–8，每個 K 用 3 個種子，每個種子 n_init=10。silhouette 衡量群內與群間距離，越高表示分隔越清楚；接近 0 表示難以分隔。ARI 比較種子間標籤一致程度，1 為完全一致，接近 0 為接近隨機一致。
選擇規則：第一個種子每群至少 20 筆的 K 中，取平均 silhouette 最大者；只作基線選擇，不代表找到最佳真實主題數。


In [ ]:
import os
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
from pathlib import Path
import pandas as pd,numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score,adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations

from pathlib import Path
OUT = Path("results")
m=pd.read_csv(OUT / "model_comparison.csv")
a=pd.read_csv(OUT / "record_review/songs_reviewed.csv")
data=m[m.source_row_id.ne(533)][['source_row_id','Track Name','artist_clean','year']].merge(a[['source_row_id','lyrics_unique_lines_v2']],on='source_row_id',validate='one_to_one').sort_values('source_row_id').reset_index(drop=True)
# Unique lines reduce chorus dominance for theme discovery. Sentiment uses both versions.
text=data.lyrics_unique_lines_v2.str.lower().str.replace(r'[^a-z\s\u0027\u2019]',' ',regex=True).str.replace(r'\s+',' ',regex=True).str.strip()
custom={'yeah','oh','ooh','uh','ah','hey','na','la','ll','ve','re','don','ain','gonna','wanna','gotta','cause','ya','woo','ha','ayy','bam','ba','da','eh','chorus','verse','refrain','intro','outro','repeat','instrumental','feat'}
vectorizer=TfidfVectorizer(max_features=3000,min_df=5,max_df=.8,ngram_range=(1,2),sublinear_tf=True,stop_words=sorted(set(ENGLISH_STOP_WORDS)|custom))
X=vectorizer.fit_transform(text)
assert (X.getnnz(axis=1)>0).all()
seeds=[1012,2026,42];metrics=[];models={}
for k in range(2,9):
 fits=[];sil=[]
 for seed in seeds:
  model=KMeans(n_clusters=k,random_state=seed,n_init=10).fit(X)
  fits.append(model);sil.append(silhouette_score(X,model.labels_,metric='cosine'))
 sizes=np.bincount(fits[0].labels_,minlength=k)
 stability=[adjusted_rand_score(x.labels_,y.labels_) for x,y in combinations(fits,2)]
 metrics.append({'k':k,'silhouette_mean':np.mean(sil),'silhouette_sd':np.std(sil),'seed_stability_ari':np.mean(stability),'smallest_cluster':sizes.min(),'largest_cluster_share':sizes.max()/len(data)})
 models[k]=fits[0]
metrics=pd.DataFrame(metrics)
# Predeclared exploration rule: highest mean silhouette among partitions with >=20 songs/cluster.
valid=metrics[metrics.smallest_cluster.ge(20)]
if valid.empty:valid=metrics
k=int(valid.sort_values(['silhouette_mean','seed_stability_ari'],ascending=False).iloc[0].k)
model=models[k];data['cluster']=model.labels_;features=np.array(vectorizer.get_feature_names_out());words=[];representatives=[]
for c in range(k):
 ix=np.flatnonzero(model.labels_==c)
 weights=model.cluster_centers_[c];top=np.argsort(weights)[::-1][:12]
 words.extend({'cluster':c,'rank':j+1,'term':features[t],'centroid_weight':weights[t]} for j,t in enumerate(top))
 sims=cosine_similarity(X[ix],model.cluster_centers_[c].reshape(1,-1)).ravel()
 order=ix[np.argsort(sims)[::-1][:5]]
 for j,i in enumerate(order):representatives.append({'cluster':c,'rank':j+1,'source_row_id':int(data.iloc[i].source_row_id),'Track Name':data.iloc[i]['Track Name'],'artist_clean':data.iloc[i].artist_clean,'cosine_to_centroid':float(cosine_similarity(X[i],model.cluster_centers_[c].reshape(1,-1))[0,0])})
out=OUT / 'clustering'
out.mkdir(parents=True, exist_ok=True)
metrics.to_csv(out/'cluster_k_comparison.csv',index=False)
pd.DataFrame(words).to_csv(out/'cluster_keywords.csv',index=False)
pd.DataFrame(representatives).to_csv(out/'cluster_representatives.csv',index=False)
data.drop(columns='lyrics_unique_lines_v2').to_csv(out/'song_clusters.csv',index=False)
print('SHAPE',X.shape,'SELECTED',k);print(metrics.to_string(index=False));print(pd.DataFrame(words).groupby('cluster').term.apply(list).to_string());print(pd.DataFrame(representatives).to_string(index=False))


### 已執行結果與解讀
在目前 927 筆資料上：K=2 平均 silhouette=0.007378、種子間平均 ARI=0.6806；第一個種子兩群為 648 與 279 筆。K=4 的 silhouette=0.004532、ARI=0.1410；K=5 的 silhouette=0.004801、ARI=0.1237。
這些結果不支持宣稱有清楚的 4 或 5 個歌詞主題。K=2 僅為此次基線的相對選擇，silhouette 仍接近 0。
群 0 代表曲含 Take Care、Wait for You、Love The Way You Lie；群 1 含 Holidae In、Promiscuous、Nasty Girl。這只是最接近各群中心的樣本，不等於已證明的主題名稱。不要因歌曲名自行標主題，也不要稱某個群等於某種情緒。
Dashboard 可展示候選 K 比較、群大小、關鍵詞和代表歌曲，並明示主題分隔薄弱。暫不以這組分群支持年度主題演變的結論。下一步可先檢查字詞表示（例如低維 LSA 或語意 embeddings）是否改善，而不是單純挑較好看的 K。


## 9 LSA 降維比較（新增）
本節接在第 8 節之後，沿用 X、data、vectorizer；若 kernel 已重新啟動，只需先重跑第 8 節，不重跑情緒模型。
LSA 在此以 TruncatedSVD 實作，再以 Normalizer 對每首歌曲的向量作 L2 正規化。比較 50、100、200 維及 K=2–8。
官方參考：https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html

比較原則：同一組標籤分別在降維空間與原始 TF-IDF 空間計算 cosine silhouette；選模型時使用原始空間的平均 silhouette，不拿不同空間的分數直接比較優劣。每個 K 用三個 K-means 種子，n_init=10。SVD 固定 seed=1012；本次 ARI 只反映 K-means 種子的穩定度，尚未測 SVD 種子或重抽樣穩定性。
K-means 仍以歐氏距離擬合，cosine silhouette 為另外的評估。候選限制為三個種子中每群都至少 20 首，再按原始空間 silhouette、ARI 排序。這是探索式選擇，沒有獨立驗證集。


In [ ]:
# 先完成第 8 節；不會呼叫情緒模型。
assert "X" in globals() and "vectorizer" in globals(), "請先執行第 8 節"
from pathlib import Path
from itertools import combinations
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from threadpoolctl import threadpool_limits
import json
threadpool_limits(limits=2)
out=Path('results/clustering')
out.mkdir(parents=True, exist_ok=True)
records=[];fits={};embeddings={};variance=[]
for dim in [50,100,200]:
 svd=TruncatedSVD(n_components=dim,random_state=1012,n_iter=7)
 Z=Normalizer().fit_transform(svd.fit_transform(X))
 embeddings[dim]=Z
 ev=float(svd.explained_variance_ratio_.sum());variance.append({'lsa_components':dim,'explained_variance_ratio':ev})
 for k in range(2,9):
  models=[];low=[];orig=[];mins=[]
  for seed in [1012,2026,42]:
   model=KMeans(n_clusters=k,random_state=seed,n_init=10).fit(Z)
   models.append(model);low.append(silhouette_score(Z,model.labels_,metric='cosine'));orig.append(silhouette_score(X,model.labels_,metric='cosine'));mins.append(int(np.bincount(model.labels_,minlength=k).min()))
  ari=np.mean([adjusted_rand_score(a.labels_,b.labels_) for a,b in combinations(models,2)])
  records.append({'lsa_components':dim,'k':k,'explained_variance_ratio':ev,'silhouette_lsa_mean':np.mean(low),'silhouette_original_mean':np.mean(orig),'seed_stability_ari':ari,'smallest_cluster_all_seeds':min(mins),'largest_cluster_share_seed1012':np.bincount(models[0].labels_).max()/len(data)})
  fits[(dim,k)]=models[0]
results=pd.DataFrame(records);results.to_csv(out/'lsa_comparison.csv',index=False)
valid=results[results.smallest_cluster_all_seeds.ge(20)]
if valid.empty:valid=results
best=valid.sort_values(['silhouette_original_mean','seed_stability_ari'],ascending=False).iloc[0]
dim,k=int(best.lsa_components),int(best.k);model=fits[(dim,k)];Z=embeddings[dim]
data['lsa_cluster']=model.labels_
words=[];reps=[]
for c in range(k):
 ix=np.flatnonzero(model.labels_==c)
 # Explain with original TF-IDF mean, never SVD dimensions presented as word topics.
 weights=np.asarray(X[ix].mean(axis=0)).ravel();order=np.argsort(weights)[::-1][:12];features=np.array(vectorizer.get_feature_names_out())
 words.extend({'cluster':c,'rank':j+1,'term':features[t],'mean_tfidf':weights[t]} for j,t in enumerate(order))
 similarities=cosine_similarity(Z[ix],model.cluster_centers_[c].reshape(1,-1)).ravel()
 for j,p in enumerate(np.argsort(similarities)[::-1][:5]):
  i=ix[p];reps.append({'cluster':c,'rank':j+1,'source_row_id':int(data.iloc[i].source_row_id),'Track Name':data.iloc[i]['Track Name'],'artist_clean':data.iloc[i].artist_clean,'cosine_to_lsa_centroid':float(similarities[p])})
data.drop(columns='lyrics_unique_lines_v2').to_csv(out/'lsa_song_clusters.csv',index=False)
pd.DataFrame(words).to_csv(out/'lsa_keywords.csv',index=False)
pd.DataFrame(reps).to_csv(out/'lsa_representatives.csv',index=False)
print('VARIANCE',variance);print('BEST',best.to_dict());print(results.sort_values('silhouette_original_mean',ascending=False).head(8).to_string(index=False));print('SIZES',data.lsa_cluster.value_counts().to_dict());print(pd.DataFrame(words).groupby('cluster').term.apply(list).to_string());print(pd.DataFrame(reps).to_string(index=False))


### LSA 已執行結果
同樣 927 首、原始 2570 維。50/100/200 維保留的 explained_variance_ratio 總和分別為 17.85%、30.42%、49.40%。這是此 SVD 的變異摘要，不是準確率，也不是語意保留率；SVD 未對資料作中心化。

此次規則選到 50 維、K=2，seed=1012 的群大小為 620 和 307；三種子中最小群至少 304。
- 原 TF-IDF K=2：原空間 silhouette=0.007378，ARI=0.6806。
- LSA 50 維 K=2：降維空間 silhouette=0.049099；原空間 silhouette=0.007461；ARI=0.9201。
只能說 K-means 種子穩定性改善，原空間分隔沒有實質提升。不要宣稱 silhouette 提升六倍或已找到兩個清楚主題。
100 維 K=2 的 ARI=0.9310 稍高，但原空間 silhouette=0.006782，沒有因最高 ARI 就改選它。
關鍵詞仍有重疊，代表歌曲僅用於人工解釋，不等於主題驗證；目前保留 cluster 0/1，不硬命名。

Dashboard 可用此比較展示：方法改善可提高穩定性，卻不保證文本自然形成明確群集。本輪先停止調整群數，主題功能以探索呈現，主要內容繼續聚焦情緒模型差異、副歌敏感度與使用者審閱案例。


## 10 目前專案狀態
情緒結果已由使用者執行，錯配排除後 927 筆；AI 初判、使用者審閱的 20 首試標僅作規則與案例探索，不是獨立測試標準。15 首正負子集上的一致比例不是全面準確率證據。首次發行年、歷史榜單抽樣與完整歌詞校驗尚未完成。
未改動原始歌詞檔。模型比較使用 uncased tokenizer 解碼後的文本，因此兩模型均失去原始大小寫線索；此限制需隨結果呈現。
